# Pandas for Data Manipulation

Source: India rows extracted from the public World Bank Life Expectancy CSV.

This notebook loads a real CSV, performs core Pandas transformations, and exports a cleaned dataset to CSV and Parquet for size comparison.

In [2]:
from io import StringIO
from pathlib import Path

import pandas as pd

source_url = "https://raw.githubusercontent.com/selva86/datasets/master/Life_Expectancy_Data.csv"
repo_root = Path.cwd()
data_dir = repo_root / "data"
data_dir.mkdir(exist_ok=True)
source_path = data_dir / "india_life_expectancy.csv"

fallback_csv = """Country,Year,Status,Life expectancy ,Adult Mortality,infant deaths,Alcohol,percentage expenditure,Hepatitis B,Measles , BMI ,under-five deaths ,Polio,Total expenditure,Diphtheria , HIV/AIDS,GDP,Population, thinness  1-19 years, thinness 5-9 years,Income composition of resources,Schooling
India,2014,Developing,68.0,184,957,3.07,86.52153895,79,79563,18.1,1200,84,4.69,85,0.2,1573.11889,1.293859294e9,26.8,27.4,0.607,11.6
India,2013,Developing,67.6,187,1000,3.11,67.67230438,7,13822,17.5,1300,82,4.53,83,0.2,1452.195373,1.27856227e9,26.8,27.5,0.599,11.5
India,2012,Developing,67.3,19,1100,3.1,64.96964491,73,18668,17.0,1400,79,4.39,82,0.2,1446.98541,1.26365852e9,26.9,27.6,0.59,11.3
India,2011,Developing,66.8,193,1100,3.0,64.6059005,44,33634,16.4,1500,79,4.33,82,0.2,1461.671957,1.24723629e8,26.9,27.7,0.58,10.8
India,2010,Developing,66.4,196,1200,2.77,57.73359864,38,31458,15.9,1600,76,4.28,79,0.2,1345.77153,1.2398691e7,27.0,27.8,0.569,10.4
"""

try:
    if source_path.exists():
        raw_df = pd.read_csv(source_path)
    else:
        world_df = pd.read_csv(source_url)
        raw_df = world_df.loc[world_df["Country"].eq("India")].copy()
        raw_df.to_csv(source_path, index=False)
except Exception:
    raw_df = pd.read_csv(StringIO(fallback_csv))
    raw_df.to_csv(source_path, index=False)

# Load the India slice from CSV into a DataFrame and inspect the core structure.
india_df = pd.read_csv(source_path)
print("Shape:", india_df.shape)
print("\nDtypes:")
print(india_df.dtypes)
print("\nHead(10):")
print(india_df.head(10).to_string(index=False))

clean_df = india_df.rename(columns=lambda column: column.strip().lower().replace(" ", "_"))
clean_df = clean_df.assign(
    decade=(clean_df["year"] // 10) * 10,
    gdp_band=pd.cut(clean_df["gdp"], bins=3, labels=["Low GDP", "Mid GDP", "High GDP"]),
)

# Filter the data to recent years so the trend analysis stays focused.
recent_df = clean_df.loc[clean_df["year"].ge(2010)].copy()
print("\nFiltered recent years (2010+):")
print(recent_df[["year", "life_expectancy", "gdp", "schooling"]].to_string(index=False))

# Group by decade to summarize the long-term trend in India's indicators.
decade_summary = clean_df.groupby("decade", as_index=False).agg(
    mean_life_expectancy=("life_expectancy", "mean"),
    mean_adult_mortality=("adult_mortality", "mean"),
    mean_gdp=("gdp", "mean"),
    mean_schooling=("schooling", "mean"),
)
print("\nDecade summary:")
print(decade_summary.to_string(index=False))

# Merge the raw observations with the decade-level summary to enrich each row with context.
merged_df = clean_df.merge(decade_summary, on="decade", how="left")
print("\nMerged sample:")
print(merged_df[["year", "decade", "life_expectancy", "mean_life_expectancy", "mean_gdp"]].head(5).to_string(index=False))

# Build a pivot table to compare life expectancy across GDP bands and decades.
pivot_df = clean_df.pivot_table(
    index="decade",
    columns="gdp_band",
    values="life_expectancy",
    aggfunc="mean",
)
print("\nPivot table: mean life expectancy by decade and GDP band")
print(pivot_df.to_string())

cleaned_path = data_dir / "w1d2_cleaned_india_life_expectancy.csv"
parquet_path = data_dir / "w1d2_cleaned_india_life_expectancy.parquet"
clean_df.to_csv(cleaned_path, index=False)
clean_df.to_parquet(parquet_path, index=False)

size_report = pd.DataFrame(
    {
        "format": ["CSV", "Parquet"],
        "path": [cleaned_path.name, parquet_path.name],
        "size_bytes": [cleaned_path.stat().st_size, parquet_path.stat().st_size],
    }
)
size_report["size_kb"] = (size_report["size_bytes"] / 1024).round(2)
print("\nExport size comparison:")
print(size_report.to_string(index=False))

smaller = size_report.sort_values("size_bytes").iloc[0]
larger = size_report.sort_values("size_bytes").iloc[-1]
print(f"\n{smaller['format']} is smaller than {larger['format']} by {larger['size_bytes'] - smaller['size_bytes']:,} bytes.")

Shape: (11, 22)

Dtypes:
Country                                str
Year                                 int64
Status                                 str
Life expectancy                    float64
Adult Mortality                      int64
infant deaths                        int64
Alcohol                            float64
percentage expenditure             float64
Hepatitis B                          int64
Measles                              int64
 BMI                               float64
under-five deaths                    int64
Polio                                int64
Total expenditure                  float64
Diphtheria                           int64
 HIV/AIDS                          float64
GDP                                float64
Population                         float64
 thinness  1-19 years              float64
 thinness 5-9 years                float64
Income composition of resources    float64
Schooling                          float64
dtype: object

Head(10):
Coun

## Notes

The cleaned India slice is written to `data/w1d2_cleaned_india_life_expectancy.csv` and `data/w1d2_cleaned_india_life_expectancy.parquet`.
For this small sample, CSV is smaller than Parquet because the dataset is tiny and Parquet adds file-format overhead.